# 01 — Data Download & Preparation

Downloads the Dogs vs. Cats dataset from Kaggle and saves it to **Google Drive** for persistence across Colab sessions.

> **Run this notebook in Google Colab** (`Runtime → Change runtime type → T4 GPU`).
>
> Dataset is saved to `MyDrive/datasets/catsvsdogs/` — subsequent sessions just mount Drive, no re-download needed.

## Environment Setup

In [ ]:
import os, json

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import userdata, drive

    # Clone the private org repo using a PAT stored in Colab Secrets
    if not os.path.exists('/content/cnn-classification-Eyad-M'):
        token = userdata.get('GITHUB_TOKEN')
        !git clone https://{token}@github.com/Eyad-M/cnn-classification-Eyad-M /content/cnn-classification-Eyad-M
    os.chdir('/content/cnn-classification-Eyad-M')

    # Mount Google Drive — data persists across sessions here
    drive.mount('/content/drive')

    # Kaggle credentials from Colab Secrets
    os.makedirs('/root/.kaggle', exist_ok=True)
    creds = {'username': userdata.get('KAGGLE_USERNAME'), 'key': userdata.get('KAGGLE_KEY')}
    with open('/root/.kaggle/kaggle.json', 'w') as f:
        json.dump(creds, f)
    os.chmod('/root/.kaggle/kaggle.json', 0o600)

DRIVE_DATA_DIR = '/content/drive/MyDrive/datasets/catsvsdogs'
DATA_DIR = DRIVE_DATA_DIR if IN_COLAB else '../data/raw'

## Install Dependencies

In [ ]:
if IN_COLAB:
    %pip install -q -r ../requirements.txt

## Download Dataset from Kaggle

In [ ]:
if not os.path.exists(DATA_DIR) or not os.listdir(DATA_DIR):
    os.makedirs(DATA_DIR, exist_ok=True)
    !kaggle datasets download -d shaunthesheep/microsoft-catsvsdogs-dataset -p "{DATA_DIR}" --unzip
    print('Download complete.')
else:
    print(f'Dataset already exists at {DATA_DIR} — skipping download.')

## Verify & Inspect Raw Files

In [ ]:
from pathlib import Path
from PIL import Image
import pandas as pd

raw_dir = Path(DATA_DIR) / 'PetImages'
records = []

for label, class_name in enumerate(['Cat', 'Dog']):
    for img_path in sorted((raw_dir / class_name).glob('*.jpg')):
        try:
            with Image.open(img_path) as img:
                img.verify()
            records.append({'path': str(img_path), 'label': label, 'class': class_name})
        except Exception:
            pass  # skip corrupted images

df = pd.DataFrame(records)
print(df['class'].value_counts())
print(f'\nTotal valid images: {len(df)}')

## Split into Train / Validation / Test

In [ ]:
from sklearn.model_selection import train_test_split

train_df, temp_df = train_test_split(df, test_size=0.3, stratify=df['label'], random_state=42)
val_df, test_df = train_test_split(temp_df, test_size=0.5, stratify=temp_df['label'], random_state=42)

print(f'Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}')

## Save Processed Data to `data/processed/`

In [ ]:
splits_dir = Path(DATA_DIR) / 'splits'
splits_dir.mkdir(exist_ok=True)

train_df.to_csv(splits_dir / 'train.csv', index=False)
val_df.to_csv(splits_dir / 'val.csv', index=False)
test_df.to_csv(splits_dir / 'test.csv', index=False)

print(f'Splits saved to {splits_dir}')